Copyright 2018-2026 AVEVA Group Limited

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
SPDX-License-Identifier: Apache-2.0

# CONNECT Iceberg REST - Time Series Data Retrieval


This notebook demonstrates zero-copy ecosystem consumption of CONNECT Virtual Tables through an open Iceberg-style flow.

# 1. Setup and Parameters

## 1.1 Setup

In [ ]:
# =========================================================
#  IMPORT LIBRARIES
# =========================================================
# requests  : REST calls to the Iceberg Catalog
# fastavro  : Iceberg manifest reading
# fsspec    : ADLS access through SAS token
# pandas    : dataframe processing
# plotly    : modern interactive visualization
# =========================================================

import json
from math import ceil

import fsspec
import pandas as pd
import plotly.express as px
import plotly.subplots as sp
import requests
from fastavro import reader


## 1.2 Connection Parameters

In [ ]:
# =========================================================
# DEFINE CONNECT ICEBERG PARAMETERS
# =========================================================
# Load parameters from appsettings.json file.
# This keeps sensitive configuration separate from the notebook.
# =========================================================

# Load configuration from appsettings.json
with open("appsettings.json", "r") as f:
    appsettings = json.load(f)

ICEBERG_ENDPOINT = appsettings["icebergEndpoint"]
WAREHOUSE = appsettings["qualifiedName"]
TOKEN = appsettings["bearerToken"]
VERIFY_SSL = False  # Demo mode only. Use a corporate CA bundle for production.
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

print("Loaded configuration from appsettings.json")
print(f"Iceberg Endpoint: {ICEBERG_ENDPOINT}")
print(f"Qualified Name: {WAREHOUSE}")

# 2. CONNECT Iceberg REST

## 2.3 Discover Iceberg Namespaces

In [ ]:
# =========================================================
# ICEBERG REST HELPER
# =========================================================
# Small helper to keep the notebook clean and readable.
# =========================================================

def iceberg_get(path: str, params: dict | None = None) -> dict:
    # GET an Iceberg REST resource and return JSON.
    url = f"{ICEBERG_ENDPOINT.rstrip('/')}/{path.lstrip('/')}"
    response = requests.get(
        url,
        headers=HEADERS,
        params=params,
        verify=VERIFY_SSL,
        timeout=60
    )
    response.raise_for_status()
    return response.json()


In [ ]:
# =========================================================
# 05 — DISCOVER ICEBERG CATALOG CONFIGURATION
# =========================================================
# The catalog returns the effective prefix used for namespaces/tables.
# =========================================================

catalog_config = iceberg_get(
    "/v1/config",
    params={"warehouse": WAREHOUSE}
)

prefix = catalog_config["overrides"]["prefix"]

print("Catalog prefix:", prefix)
print("Available endpoints:")
for endpoint in catalog_config["endpoints"]:
    print(" -", endpoint)


## 2.4 Discover Iceberg Tables

In [ ]:
# =========================================================
# DISCOVER NAMESPACES AND TABLES
# =========================================================
# Namespaces are equivalent to logical schemas.
# Tables are Iceberg table objects exposed by CONNECT.
# =========================================================

namespaces_payload = iceberg_get(f"/v1/{prefix}/namespaces")
namespaces = [tuple(ns) for ns in namespaces_payload["namespaces"]]

print("Namespaces:", namespaces)

namespace = namespaces[0][0]

tables_payload = iceberg_get(f"/v1/{prefix}/namespaces/{namespace}/tables")
raw_tables = tables_payload.get("identifiers") or tables_payload.get("tables") or []
tables = [t["name"] if isinstance(t, dict) else t[-1] for t in raw_tables]

print("Selected namespace:", namespace)
print("Tables:", tables)


## 2.5 Select Iceberg Table

In [ ]:
# Modify next line to select table of  from list generated by the previous cell
table_name = tables[0]
# Modify next line to specify whether table is narrow or wide
table_mode = "narrow"
print("Selected table:", table_name)
print("Table is", table_mode)

## 2.6 Retrieve Iceberg Metadata

In [ ]:
# =========================================================
# LOAD ICEBERG TABLE METADATA
# =========================================================
# This returns:
# - Iceberg schema
# - current snapshot
# - manifest list
# - temporary ADLS SAS token for data access
# =========================================================

table_payload = iceberg_get(
    f"/v1/{prefix}/namespaces/{namespace}/tables/{table_name}"
)

metadata = table_payload["metadata"]
config = table_payload["config"]

manifest_list_abfss = metadata["snapshots"][0]["manifest-list"]
record_count = metadata["snapshots"][0]["summary"].get("total-records")

print("Table:", table_name)
print("Records:", f"{int(record_count):,}" if record_count else "unknown")
print("Manifest list:", manifest_list_abfss)


# 3. Read and Visualize CONNECT data

## 3.1 Read ADLS Parquet Data

In [ ]:
# =========================================================
# CREATE ADLS FILESYSTEM FROM VENDED SAS TOKEN
# =========================================================
# The Iceberg endpoint provides temporary scoped access to ADLS.
# This is the key zero-copy federation mechanism.
# =========================================================

sas_key = next(k for k in config if k.startswith("adls.sas-token."))
storage_account = sas_key.split(".")[2]
sas_token = config[sas_key]

fs = fsspec.filesystem(
    "abfs",
    account_name=storage_account,
    sas_token=sas_token
)

print("Storage account:", storage_account)
print("SAS token available:", bool(sas_token))

In [ ]:
# =========================================================
# RESOLVE ICEBERG MANIFESTS TO PARQUET DATA FILES
# =========================================================
# Iceberg metadata points to manifest files.
# Manifest files point to the physical Parquet data files.
# =========================================================

with fs.open(manifest_list_abfss, "rb") as f:
    manifest_records = list(reader(f))

manifest_paths = [record["manifest_path"] for record in manifest_records]
print("Manifest files:", len(manifest_paths))

all_data_file_paths = []
for manifest_path in manifest_paths:
    with fs.open(manifest_path, "rb") as f:
        data_file_records = list(reader(f))
    all_data_file_paths.extend(
        record["data_file"]["file_path"]
        for record in data_file_records
    )

print("Parquet data files:", len(all_data_file_paths))
print(all_data_file_paths[0])

## 3.2 Prepare Time Series Model

In [ ]:
# =========================================================
# READ PARQUET DATA INTO PANDAS
# =========================================================
# For the demo table, one Parquet data file contains the full dataset.
# Extract column names from Iceberg schema instead of hardcoding.
# =========================================================

# Get column names from Iceberg schema
schema = metadata["schemas"]
column_names = [field["name"] for field in schema[0]["fields"]]

df = pd.concat(
    [pd.read_parquet(path, filesystem=fs) for path in all_data_file_paths],
    ignore_index=True
)

print("Raw shape:", df.shape)
print("Raw columns:", df.columns.tolist())
print("Column names from Iceberg schema:", column_names)

# Rename columns using the schema from Iceberg
df.columns = column_names

df.head(10)

In [ ]:
# =========================================================
# PREPARE DATA FOR VISUALIZATION
# =========================================================
# Automatically identify:
# - Timestamp column (first datetime-like column)
# - Value columns (numeric columns)
# - Dimension columns (categorical columns)
# =========================================================

# Identify timestamp column
timestamp_col = None
for col in df.columns:
    if pd.api.types.is_datetime64_any_dtype(df[col]):
        timestamp_col = col
        break

# Identify numeric (value) columns
value_cols = df.select_dtypes(include=['number']).columns.tolist()

# Identify categorical (dimension) columns
dimension_cols = [col for col in df.columns 
                  if col != timestamp_col and col not in value_cols]

print(f"Timestamp column: {timestamp_col}")
print(f"Value columns: {value_cols}")
print(f"Dimension columns: {dimension_cols}")


In [ ]:
# =========================================================
# PLOT COLUMNS AGAINST TIMESTAMP
# =========================================================
# Wide mode: one subplot per metric/dimension column
# Narrow mode: one subplot per unique value in Field column
# =========================================================

# Ensure chronological order before plotting
df_plot = df.sort_values(by=timestamp_col).reset_index(drop=True)

if table_mode.lower() == "narrow" and {"Field", "Value"}.issubset(df_plot.columns):
    # Create one plot per unique field name in narrow tables.
    field_values = sorted(df_plot["Field"].dropna().astype(str).unique().tolist())
    num_plots = len(field_values)
    cols_per_row = 2
    num_rows = ceil(num_plots / cols_per_row) if num_plots > 0 else 1

    fig = sp.make_subplots(
        rows=num_rows,
        cols=cols_per_row,
        subplot_titles=[f"{field} vs {timestamp_col}" for field in field_values],
        specs=[[{"secondary_y": False} for _ in range(cols_per_row)] for _ in range(num_rows)]
    )

    for idx, field_name in enumerate(field_values):
        row = idx // cols_per_row + 1
        col_pos = idx % cols_per_row + 1

        field_df = df_plot[df_plot["Field"].astype(str) == field_name].copy()
        field_df["Value_numeric"] = pd.to_numeric(field_df["Value"], errors="coerce")

        if field_df["Value_numeric"].notna().any():
            trace = px.line(field_df, x=timestamp_col, y="Value_numeric").data[0]
        else:
            # Keep string/categorical value series as discrete markers.
            trace = px.scatter(field_df, x=timestamp_col, y="Value").data[0]
            trace.update(mode="markers", marker={"size": 5, "opacity": 0.75})

        fig.add_trace(trace, row=row, col=col_pos)
        fig.update_xaxes(title_text=timestamp_col, row=row, col=col_pos)
        fig.update_yaxes(title_text="Value", row=row, col=col_pos)

    chart_title = f"Narrow Table: Field Values vs {timestamp_col}"
else:
    # Wide-table behavior: one plot per value/dimension column.
    plot_cols = value_cols + dimension_cols
    num_plots = len(plot_cols)
    cols_per_row = 2
    num_rows = ceil(num_plots / cols_per_row) if num_plots > 0 else 1

    fig = sp.make_subplots(
        rows=num_rows,
        cols=cols_per_row,
        subplot_titles=[f"{col} vs {timestamp_col}" for col in plot_cols],
        specs=[[{"secondary_y": False} for _ in range(cols_per_row)] for _ in range(num_rows)]
    )

    for idx, series_col in enumerate(plot_cols):
        row = idx // cols_per_row + 1
        col_pos = idx % cols_per_row + 1

        if series_col in value_cols:
            trace = px.line(df_plot, x=timestamp_col, y=series_col).data[0]
        else:
            trace = px.scatter(df_plot, x=timestamp_col, y=series_col).data[0]
            trace.update(mode="markers", marker={"size": 5, "opacity": 0.75})

        fig.add_trace(trace, row=row, col=col_pos)
        fig.update_xaxes(title_text=timestamp_col, row=row, col=col_pos)
        fig.update_yaxes(title_text=series_col, row=row, col=col_pos)

    chart_title = f"Metrics and Dimensions vs {timestamp_col}"

fig.update_layout(
    template="plotly_dark",
    height=300 * num_rows,
    title_text=chart_title,
    showlegend=False,
    paper_bgcolor="#1e1e1e",
    plot_bgcolor="#1e1e1e"
)

fig.show()
